# کاهش ابعاد و مصورسازی Embeddingها با PCA

این ماژول برای بارگذاری بردارهای ویژگی استخراج‌شده توسط مدل **DINOv2**، کاهش ابعاد آن‌ها با استفاده از الگوریتم **Principal Component Analysis (PCA)** و نمایش تعاملی تصاویر در فضای سه‌بعدی طراحی شده است.

## ۱. معرفی

بردارهای Embedding استخراج‌شده توسط DINOv2 دارای ابعاد بالایی هستند. PCA با تبدیل این بردارها به مجموعه‌ای از مؤلفه‌های اصلی، امکان کاهش ابعاد و بررسی ساختار داده‌ها را فراهم می‌کند.

این ماژول شامل سه تابع اصلی است:

- `load_embeddings()`: بارگذاری Embeddingها از فایل Parquet، بررسی اعتبار بردارها و ساخت ماتریس ویژگی.
- `run_pca()`: اجرای PCA و تولید مؤلفه‌های اصلی.
- `plot_pca_3d()`: نمایش تعاملی داده‌ها در فضای سه‌بعدی و نمایش تصویر اصلی هنگام قرار گرفتن نشانگر ماوس روی هر نقطه.

## ۲. مبانی ریاضی PCA

فرض می‌کنیم ماتریس داده‌ها به شکل زیر باشد:

$$
X \in \mathbb{R}^{n \times d}
$$

که در آن \(n\) تعداد تصاویر و \(d\) تعداد ویژگی‌های هر Embedding است.

ابتدا داده‌ها مرکزدهی می‌شوند:

$$
X_c = X - \mu
$$

سپس ماتریس کوواریانس محاسبه می‌شود:

$$
C = \frac{1}{n-1} X_c^T X_c
$$

با تجزیه مقادیر ویژه، بردارها و مقادیر ویژه به دست می‌آیند:

$$
C v_i = \lambda_i v_i
$$

بردارهای ویژه متناظر با بزرگ‌ترین مقادیر ویژه، مؤلفه‌های اصلی را تشکیل می‌دهند.

در نهایت، داده‌ها به فضای با ابعاد کمتر منتقل می‌شوند:

$$
Z = X_c W_k
$$

که در آن \(W_k\) شامل \(k\) بردار ویژه اول است.

**نسبت واریانس توضیح داده‌شده (Explained Variance Ratio):**

$$
EVR_i = \frac{\lambda_i}{\sum_j \lambda_j}
$$

این مقدار نشان می‌دهد هر مؤلفه اصلی چه نسبتی از واریانس کل داده‌ها را حفظ می‌کند.

## ۳. مراحل اجرا

1. بارگذاری Embeddingهای استخراج‌شده از DINOv2.
2. حذف بردارهای نامعتبر و ناسازگار از نظر ابعاد.
3. نرمال‌سازی L2 در صورت نیاز.
4. اجرای PCA و کاهش ابعاد به سه مؤلفه اصلی.
5. نمایش داده‌ها در فضای سه‌بعدی با استفاده از Plotly.
6. نمایش تصویر اصلی هر نمونه با قرار گرفتن نشانگر ماوس روی نقطه متناظر.

## ۴. ورودی و خروجی

**ورودی:**
- فایل Parquet شامل مسیر تصاویر و بردارهای Embedding.
- ستون‌های اختیاری شامل شناسه تصویر و برچسب دسته‌بندی.

**خروجی:**
- ماتریس ویژگی کاهش‌یافته شامل `PC1`، `PC2` و `PC3`.
- مدل آموزش‌دیده PCA.
- نمودار سه‌بعدی تعاملی همراه با قابلیت مشاهده تصاویر اصلی.

## ۵. کتابخانه‌های مورد استفاده

- NumPy و Pandas: پردازش و آماده‌سازی داده‌ها.
- Scikit-learn: اجرای PCA و نرمال‌سازی بردارها.
- Plotly: رسم نمودار سه‌بعدی تعاملی.
- Pillow (PIL): بارگذاری و نمایش تصاویر.
- ipywidgets: ایجاد رابط تعاملی در Jupyter Notebook.

In [1]:

from pathlib import Path
import sys

# اگر نوت‌بوک داخل پوشه notebooks است، ریشه پروژه یک سطح بالاتر است.
PROJECT_ROOT = Path.cwd().parent.resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

from pca import load_embeddings, run_pca, plot_pca_3d

In [2]:
EMBEDDINGS_PATH = PROJECT_ROOT / 'data/embeddings/images_dinov2.parquet'
PCA_OUTPUT_PATH = PROJECT_ROOT / 'data/embeddings/images_dinov2_pca3.parquet'

print('Project:', PROJECT_ROOT)
print('Embeddings:', EMBEDDINGS_PATH)
assert EMBEDDINGS_PATH.exists(), f'File not found: {EMBEDDINGS_PATH}'

Project: D:\Study\SUT-CS\Books\تحلیل داده\پروژه\phase-2
Embeddings: D:\Study\SUT-CS\Books\تحلیل داده\پروژه\phase-2\data\embeddings\images_dinov2.parquet


In [3]:
df, X, embedding_col = load_embeddings(
    EMBEDDINGS_PATH,
    image_path_col='image_path',
    id_col='image_id',
)
display(df.head())
print('Matrix shape:', X.shape)

Loaded 28,132 vectors | dimension=384 | column='embedding'


,image_id,observation_uuid,image_path,animal_present,embedding
0,001cc818-ac3e-4750-979d-4d2d9d8358f7,001cc818-ac3e-4750-979d-4d2d9d8358f7,d:\Study\SUT-CS\Books\تحلیل داده\پروژه\phase-2...,None,"[-0.04448, -0.025397, -0.149815, -0.006841, -0..."
1,0096e64b-71cf-42d9-b8ad-519bcacd14a8,0096e64b-71cf-42d9-b8ad-519bcacd14a8,d:\Study\SUT-CS\Books\تحلیل داده\پروژه\phase-2...,None,"[-0.022113, -0.032662, -0.120739, -0.02755, -0..."
2,01057a20-6a12-4d43-8a0d-edaead5284e5,01057a20-6a12-4d43-8a0d-edaead5284e5,d:\Study\SUT-CS\Books\تحلیل داده\پروژه\phase-2...,None,"[-0.018675, -0.094978, -0.006159, -0.050003, 0..."
3,014f362c-2e4e-4e51-aa4e-76b6d2315a32,014f362c-2e4e-4e51-aa4e-76b6d2315a32,d:\Study\SUT-CS\Books\تحلیل داده\پروژه\phase-2...,None,"[0.027847, -0.063047, -0.045055, -0.014622, -0..."
4,0162c51c-4627-417c-b6f2-b75f3164ba69,0162c51c-4627-417c-b6f2-b75f3164ba69,d:\Study\SUT-CS\Books\تحلیل داده\پروژه\phase-2...,None,"[-0.007697, -0.036371, 0.025362, -0.06874, -0...."


Matrix shape: (28132, 384)


In [4]:
# اجرای PCA سه‌بعدی
pca_df, pca_model = run_pca(df, X, n_components=3)
pca_df[[c for c in ['image_id', 'image_path', 'PC1', 'PC2', 'PC3'] if c in pca_df.columns]].head()

Explained variance:
  PC1: 4.81%
  PC2: 3.97%
  PC3: 2.90%
  Total: 11.69%


,image_id,image_path,PC1,PC2,PC3
0,001cc818-ac3e-4750-979d-4d2d9d8358f7,d:\Study\SUT-CS\Books\تحلیل داده\پروژه\phase-2...,0.338000,0.107004,0.017662
1,0096e64b-71cf-42d9-b8ad-519bcacd14a8,d:\Study\SUT-CS\Books\تحلیل داده\پروژه\phase-2...,0.378286,0.076375,-0.002990
2,01057a20-6a12-4d43-8a0d-edaead5284e5,d:\Study\SUT-CS\Books\تحلیل داده\پروژه\phase-2...,0.417835,0.002369,0.028816
3,014f362c-2e4e-4e51-aa4e-76b6d2315a32,d:\Study\SUT-CS\Books\تحلیل داده\پروژه\phase-2...,0.480640,-0.008728,0.087000
4,0162c51c-4627-417c-b6f2-b75f3164ba69,d:\Study\SUT-CS\Books\تحلیل داده\پروژه\phase-2...,0.417201,0.061330,0.030287


خیلی کمه اطلاعات موجودی که بهمون میده

In [5]:
# ذخیره مختصات PCA برای استفاده‌های بعدی
PCA_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
pca_df.to_parquet(PCA_OUTPUT_PATH, index=False)
print(f'Saved: {PCA_OUTPUT_PATH}')

Saved: D:\Study\SUT-CS\Books\تحلیل داده\پروژه\phase-2\data\embeddings\images_dinov2_pca3.parquet


In [6]:
import pandas as pd

observations_path = PROJECT_ROOT / "data/processed/observations.parquet"

observations_df = pd.read_parquet(observations_path)

# اتصال تصاویر به مشاهدات
merged_df = pca_df.merge(
    observations_df[
        ["uuid", "taxon_class_name", "common_name"]
    ],
    left_on="observation_uuid",
    right_on="uuid",
    how="left"
)

print(merged_df["taxon_class_name"].value_counts(dropna=False))

taxon_class_name
Aves        18811
Reptilia     4344
Mammalia     3334
Amphibia     1643
Name: count, dtype: int64


In [7]:
TARGET_CLASSES = [
    "Mammalia",
    "Reptilia",
    "Aves",
    "Amphibia"
]

sample_df = (
    merged_df[
        merged_df["taxon_class_name"].isin(TARGET_CLASSES)
    ]
    .groupby("taxon_class_name", group_keys=False)
    .sample(
        n=250,
        random_state=42
    )
    .reset_index(drop=True)
)

print(sample_df["taxon_class_name"].value_counts())
print("Total images:", len(sample_df))

taxon_class_name
Amphibia    250
Aves        250
Mammalia    250
Reptilia    250
Name: count, dtype: int64
Total images: 1000


In [8]:
fig, image_output = plot_pca_3d(
    sample_df,
    project_root=PROJECT_ROOT,
    image_path_col="image_path",
    id_col="image_id",
    point_size=5,
)

    'data': [{'customdata': [['d:\\Study\\SUT-CS\\Books\\تحلیل داده\\پروژه\\phas…